In [ ]:
import kagglehub
import os
import pandas as pd
#ANON_ANON_[REDACTED]

# Download latest version

path = kagglehub.dataset_download("mohammad2012191/q3-ka-ai-2026")



print("Path to dataset files:", path)

In [ ]:
# Task 1: Write your code here:
csv_path = os.path.join(path, "Q3_data.csv")

df = pd.read_csv(csv_path)




In [ ]:
# Task 2: Write your code here:

df.head()


In [ ]:
# Task 3: Write your code here:
df.info()

In [ ]:
# Task 4: Write your code here:

df.describe()


In [ ]:
# Task 1: Write your code here:
df.isnull().sum()


# Identify numeric and categorical columns
numeric_cols = df.select_dtypes(include=['int64', 'float64']).columns
categorical_cols = df.select_dtypes(include=['object']).columns
# we will Fill missing numeric values with the mean
for col in numeric_cols:
    df[col] = df[col].fillna(df[col].mean())
    #we will Fill missing with categorical values with the most frequent value (mode)
for col in categorical_cols:
    df[col] = df[col].fillna(df[col].mode()[0])

    # Verify that there are no missing values left
df.isnull().sum()




In [ ]:
# Task 2: Write your code here:
# Do we have duplicate samples?
def check_duplicates(df):
  duplicates = df.duplicated().sum()
  print(f"Number of Duplicate Samples: {duplicates}")
  if duplicates > 0:
    print("Dropping Duplicates...")
    df.drop_duplicates(inplace=True)
    print("Duplicates Dropped.")
  else:
    print("No Duplicate Samples Found.")

check_duplicates(df)
df.describe()

In [ ]:
# Task 3: Write your code here:
df.dtypes
# no categorical encoding.

In [ ]:
# Task 4: Write your code here:
from sklearn.preprocessing import StandardScaler

# Separate
X = df.drop(columns=['Target'])
y = df['Target']


scaler = StandardScaler()

X_scaled = scaler.fit_transform(X)
X_scaled = pd.DataFrame(X_scaled, columns=X.columns)
X_scaled.head()


In [ ]:
# Task 5: Write your code here:
df['Target'].value_counts(normalize=True)

In [ ]:
# Task 1: Write your code here:
# Separate features and target variable
X = df.drop(columns=['Target'])
y = df['Target']

In [ ]:
!pip install catboost


In [ ]:
from catboost import CatBoostClassifier
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score
import numpy as np
import pandas as pd

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
f1_scores = []

X_data = X_scaled if 'X_scaled' in globals() else X
if not isinstance(X_data, pd.DataFrame):
    X_data = pd.DataFrame(X_data)

for train_idx, val_idx in skf.split(X_data, y):
    X_train_fold = X_data.iloc[train_idx]
    X_val_fold   = X_data.iloc[val_idx]
    y_train_fold = y.iloc[train_idx]
    y_val_fold   = y.iloc[val_idx]

    model = CatBoostClassifier(
        iterations=200,
        depth=6,
        learning_rate=0.1,
        random_state=42,
        verbose=0
    )

    model.fit(X_train_fold, y_train_fold)
    y_pred = model.predict(X_val_fold)
    f1_scores.append(f1_score(y_val_fold, y_pred))

print("Average F1 Score across folds:", np.mean(f1_scores))

In [ ]:
# Task 1: Write your code here:
from catboost import CatBoostClassifier

final_model = CatBoostClassifier(random_state=42, verbose=0)
final_model.fit(X_data, y)
import matplotlib.pyplot as plt
import numpy as np

importances = final_model.get_feature_importance()
feature_names = X_data.columns

indices = np.argsort(importances)[::-1]

# Plot feature importance
plt.figure(figsize=(10, 6))
plt.bar(range(len(importances)), importances[indices])
plt.xticks(range(len(importances)), feature_names[indices], rotation=90)
plt.xlabel("Features")
plt.ylabel("Importance Score")
plt.title("Feature Importance (Golden Feature Analysis)")
plt.tight_layout()
plt.show()


In [ ]:
# Task 2: Write your code here:
importances = final_model.get_feature_importance()
feature_names = X_data.columns

# Find  index
golden_feature_idx = importances.argmax()
golden_feature = feature_names[golden_feature_idx]
print("Golden Feature:", golden_feature)